# Lab 2 — Task 1: clasificación de géneros musicales con una CNN

En este notebook entrenamos una **red neuronal convolucional (CNN) desde cero** para clasificar imágenes de espectrogramas de música.

El objetivo es comparar una configuración base con experimentos que cambian la profundidad de la red o un hiperparámetro cada vez.

**Plan de trabajo**
1. Cargar las imágenes y separar los datos.
2. Definir las 17 configuraciones experimentales.
3. Entrenar cada configuración durante un máximo de **50 épocas completas**.
4. Comparar los resultados usando únicamente la accuracy de validación.
5. Evaluar una sola vez en test el modelo seleccionado.

La Task 2 (VGG16) no forma parte de este notebook.

## 1. Importaciones y configuración general

Importamos las librerías necesarias y fijamos las semillas aleatorias para facilitar la reproducibilidad. Ajusta `DATA_DIR` si las imágenes están en otra carpeta.

In [ ]:
# Si falta alguna librería, ejecuta esta línea en una celda aparte:
# %pip install tensorflow scikit-learn matplotlib pandas

import os
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Semillas aleatorias para que la partición y la inicialización sean reproducibles.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Ruta a las imágenes: una subcarpeta por género musical.
DATA_DIR = "Data/images_original"

# Parámetros de entrenamiento comunes a todos los experimentos.
BATCH_SIZE = 32
MAX_EPOCHS = 50

print("TensorFlow:", tf.__version__)
print("Épocas por configuración:", MAX_EPOCHS)

## 2. Cargar las imágenes y dividir los datos

Cada subcarpeta de `images_original` debe representar un género musical. Primero reservamos el 20 % de las imágenes para test. El 80 % restante se divide en entrenamiento y validación.

La partición final es aproximadamente **64 % entrenamiento, 16 % validación y 20 % test**. La división es estratificada, de modo que cada conjunto conserve una proporción similar de cada género. El conjunto de test no se utiliza para elegir configuraciones.

In [ ]:
# Comprobar que la ruta existe antes de leer los archivos.
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"No encuentro la carpeta '{DATA_DIR}'. Cambia DATA_DIR para indicar la ruta correcta."
    )

# Obtener los nombres de las subcarpetas, ordenados alfabéticamente.
class_names = sorted(
    folder for folder in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, folder))
)

# Recoger las rutas de las imágenes y su etiqueta numérica.
valid_extensions = (".png", ".jpg", ".jpeg", ".bmp")
paths = []
labels = []

for label, genre in enumerate(class_names):
    genre_folder = os.path.join(DATA_DIR, genre)

    for filename in sorted(os.listdir(genre_folder)):
        if filename.lower().endswith(valid_extensions):
            paths.append(os.path.join(genre_folder, filename))
            labels.append(label)

paths = np.asarray(paths)
labels = np.asarray(labels)

if len(paths) == 0:
    raise ValueError("No se encontraron imágenes. Comprueba DATA_DIR y la estructura de carpetas.")

if len(class_names) < 2:
    raise ValueError("Se necesitan al menos dos carpetas de géneros para clasificar.")

print("Géneros:", class_names)
print("Total de imágenes:", len(paths))

# Separar primero el conjunto de test (20 %).
dev_paths, test_paths, dev_labels, test_labels = train_test_split(
    paths,
    labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

# Dividir el 80 % restante en entrenamiento y validación.
train_paths, val_paths, train_labels, val_labels = train_test_split(
    dev_paths,
    dev_labels,
    test_size=0.20,
    random_state=SEED,
    stratify=dev_labels
)

print(f"Entrenamiento: {len(train_paths)}")
print(f"Validación:    {len(val_paths)}")
print(f"Test:          {len(test_paths)}")
print("El conjunto de test queda reservado hasta el final.")

## 3. Preparar los datasets de imágenes

Las imágenes se leen desde el disco y se redimensionan al tamaño que necesite cada configuración. Las etiquetas son números enteros, por lo que usaremos la función de pérdida `sparse_categorical_crossentropy`.

Solo barajamos las imágenes de entrenamiento. Validación y test mantienen su orden para que las predicciones puedan compararse con las etiquetas correctas.

In [ ]:
# Leer una imagen, convertirla a RGB y cambiar su resolución.
def load_image(path, label, size):
    image_bytes = tf.io.read_file(path)
    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, size)
    return image, label


# Crear un dataset por lotes para entrenar o evaluar el modelo.
def make_dataset(file_paths, file_labels, size, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, file_labels))

    # Barajar solo entrenamiento; la semilla mantiene el proceso reproducible.
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(file_paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    # Cargar y redimensionar imágenes en paralelo.
    dataset = dataset.map(
        lambda path, label: load_image(path, label, size),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Agrupar imágenes y preparar el siguiente lote mientras se entrena.
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Mostrar algunos ejemplos para comprobar que las imágenes y etiquetas se leen bien.
preview_ds = make_dataset(
    train_paths, train_labels, size=(128, 192), training=False
)

for images, labels_batch in preview_ds.take(1):
    plt.figure(figsize=(10, 3))
    for i in range(min(4, len(images))):
        plt.subplot(1, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(class_names[int(labels_batch[i].numpy())])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 4. Definir las 17 configuraciones experimentales

`BASE0` es la configuración de referencia. Cada experimento de profundidad cambia la secuencia de filtros; cada experimento de factores cambia **solo el parámetro indicado** respecto a `BASE0`.

El experimento de resolución `192×288` se coloca al final porque requiere más memoria y tiempo de cálculo.

In [ ]:
# Configuración base. Estos valores se mantienen exactamente como se han especificado.
BASE0 = dict(
    filters=(32, 64, 128),
    kernel=(3, 3),
    dropout=0.3,
    augment=True,
    lr=1e-3,
    size=(128, 192)
)

# Experimentos de profundidad: solo cambia la secuencia de filtros.
DEPTH_EXPERIMENTS = {
    "2conv_16-32":             dict(filters=(16, 32)),
    "3conv_16-32-64":          dict(filters=(16, 32, 64)),
    "3conv_32-64-128":         dict(filters=(32, 64, 128)),
    "4conv_16-32-64-128":      dict(filters=(16, 32, 64, 128)),
    "4conv_32-64-128-256":     dict(filters=(32, 64, 128, 256)),
    "5conv_32-64-128-256-256": dict(filters=(32, 64, 128, 256, 256)),
}

# Experimentos de factores: cada uno cambia una propiedad de BASE0.
FACTOR_EXPERIMENTS = {
    "dropout_0.0":     dict(dropout=0.0),
    "dropout_0.5":     dict(dropout=0.5),
    "no_augment":      dict(augment=False),
    "lr_3e-4":         dict(lr=3e-4),
    "lr_3e-3":         dict(lr=3e-3),
    "kernel_5x5":      dict(kernel=(5, 5)),
    "kernel_7x3_freq": dict(kernel=(7, 3)),  # Kernel más alto en frecuencia.
    "kernel_3x7_time": dict(kernel=(3, 7)),  # Kernel más ancho en tiempo.
    "size_64x96":      dict(size=(64, 96)),
    "size_192x288":   dict(size=(192, 288)),
}

# Combinar BASE0 con los cambios de cada experimento.
def merge_config(base, changes):
    config = base.copy()
    config.update(changes)
    return config

configurations = {"BASE0": BASE0.copy()}

for name, changes in DEPTH_EXPERIMENTS.items():
    configurations[name] = merge_config(BASE0, changes)

for name, changes in FACTOR_EXPERIMENTS.items():
    configurations[name] = merge_config(BASE0, changes)

# Mover la resolución más grande al final de la lista.
largest_resolution_config = configurations.pop("size_192x288")
configurations["size_192x288"] = largest_resolution_config

# Mostrar la tabla antes de empezar para comprobar los parámetros.
config_table = pd.DataFrame.from_dict(configurations, orient="index")
config_table.index.name = "Configuration"
config_table = config_table.reset_index()

print("Número total de configuraciones:", len(configurations))
display(config_table)

assert len(configurations) == 17, "Debe haber exactamente 17 configuraciones."

## 5. Construir la CNN

La red contiene una serie de bloques convolucionales. Cada bloque aplica una convolución, una activación ReLU y una reducción de resolución con max pooling.

Después usamos global average pooling, una capa densa y dropout opcional antes de la salida. La capa final tiene una neurona por género y utiliza softmax para producir probabilidades.

La normalización de píxeles y el data augmentation se aplican dentro del modelo. Keras activa las capas de augmentation durante el entrenamiento, no durante la evaluación.

In [ ]:
# Crear el modelo a partir de una configuración.
def generate_model(num_classes, config):
    height, width = config["size"]
    inputs = tf.keras.Input(shape=(height, width, 3))
    x = inputs

    # Aumentar ligeramente los datos si la configuración lo solicita.
    # Estas capas solo modifican las imágenes durante el entrenamiento.
    if config["augment"]:
        x = tf.keras.layers.RandomTranslation(0.05, 0.05)(x)
        x = tf.keras.layers.RandomZoom(0.1)(x)
        x = tf.keras.layers.RandomContrast(0.1)(x)

    # Convertir los píxeles de [0, 255] a [0, 1].
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(x)

    # Añadir los bloques convolucionales definidos por la lista de filtros.
    for n_filters in config["filters"]:
        x = tf.keras.layers.Conv2D(
            filters=n_filters,
            kernel_size=config["kernel"],
            padding="same",
            activation="relu"
        )(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    # Resumir los mapas de características y clasificar el género.
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)

    # Dropout ayuda a reducir el sobreajuste si su valor es mayor que cero.
    if config["dropout"] > 0:
        x = tf.keras.layers.Dropout(config["dropout"])(x)

    outputs = tf.keras.layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Adam usa el learning rate definido para cada experimento.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config["lr"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# Guardar los pesos de la época con mejor accuracy de validación,
# pero seguir entrenando hasta completar las 50 épocas.
class SaveBestWeights(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.best_val_accuracy = -1.0
        self.best_weights = None
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        val_accuracy = (logs or {}).get("val_accuracy")

        if val_accuracy is not None and val_accuracy > self.best_val_accuracy:
            self.best_val_accuracy = float(val_accuracy)
            self.best_weights = [
                weight.copy() for weight in self.model.get_weights()
            ]
            self.best_epoch = epoch + 1

## 6. Entrenar y comparar todas las configuraciones

Cada configuración se entrena durante **50 épocas completas**, sin early stopping. Se registra la accuracy de entrenamiento y validación de cada época, y se guardan los pesos de la época con mejor accuracy de validación.

Al terminar cada experimento se libera memoria antes de crear el siguiente modelo. La configuración ganadora se decide usando únicamente la validación; no se consulta el test durante la comparación.

In [ ]:
# Diccionarios donde guardaremos resultados y el mejor modelo.
num_classes = len(class_names)
experiment_results = {}

best_name = None
best_val_accuracy = -1.0
best_weights = None
best_config = None

# Entrenar una configuración cada vez, en el orden de la tabla.
for index, (name, config) in enumerate(configurations.items(), start=1):
    print("\n" + "=" * 75)
    print(f"[{index}/{len(configurations)}] Entrenando: {name}")
    print("Parámetros:", config)
    print(f"Épocas: {MAX_EPOCHS}")
    print("=" * 75)

    # Limpiar recursos y reiniciar la semilla antes de cada experimento.
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)

    # Crear los datasets con la resolución de esta configuración.
    train_ds = make_dataset(
        train_paths, train_labels, config["size"], training=True
    )
    val_ds = make_dataset(
        val_paths, val_labels, config["size"], training=False
    )

    # Crear y entrenar la red durante las 50 épocas.
    candidate = generate_model(num_classes, config)
    save_best = SaveBestWeights()

    history = candidate.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        shuffle=False,  # El barajado ya se hace en train_ds.
        callbacks=[save_best],
        verbose=1
    )

    # Guardar las curvas completas para poder compararlas después.
    run = {
        "config": config.copy(),
        "history": {
            key: list(values)
            for key, values in history.history.items()
        },
        "val_accuracy": save_best.best_val_accuracy,
        "best_epoch": save_best.best_epoch,
        "epochs": len(history.history["loss"]),
        "train_accuracy": float(
            history.history["accuracy"][save_best.best_epoch - 1]
        )
    }
    experiment_results[name] = run

    print(
        f"Mejor val accuracy: {run['val_accuracy']:.4f} "
        f"(época {run['best_epoch']})"
    )

    # Si mejora el resultado anterior, guardar sus pesos y configuración.
    if run["val_accuracy"] > best_val_accuracy:
        best_val_accuracy = run["val_accuracy"]
        best_name = name
        best_config = config.copy()
        best_weights = [
            weight.copy() for weight in save_best.best_weights
        ]

    # Eliminar objetos temporales para no acumular modelos en memoria.
    del candidate, history, train_ds, val_ds, save_best
    tf.keras.backend.clear_session()
    gc.collect()

print("\n" + "=" * 75)
print("TODOS LOS EXPERIMENTOS HAN TERMINADO")
print("Mejor configuración:", best_name)
print(f"Mejor validation accuracy: {best_val_accuracy:.4f}")
print("Parámetros elegidos:", best_config)

## 7. Tabla comparativa de resultados

Ordenamos las configuraciones de mayor a menor accuracy de validación. La columna `Best epoch` indica en qué época se obtuvieron los mejores pesos; todos los experimentos han completado las 50 épocas.

In [ ]:
# Crear una fila de tabla por configuración.
results_rows = []

for name, run in experiment_results.items():
    results_rows.append({
        "Configuration": name,
        "Train accuracy": run["train_accuracy"],
        "Validation accuracy": run["val_accuracy"],
        "Best epoch": run["best_epoch"],
        "Epochs completed": run["epochs"],
        "Parameters": str(run["config"])
    })

results_df = pd.DataFrame(results_rows)

# Mostrar primero la configuración con mejor resultado de validación.
results_df = results_df.sort_values(
    "Validation accuracy",
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

print("Configuración seleccionada:", best_name)
print(f"Validation accuracy: {best_val_accuracy:.4f}")

## 8. Evaluación final en el conjunto de test

Esta es la única sección que utiliza test para evaluar el rendimiento final. Reconstruimos la configuración ganadora, cargamos sus mejores pesos de validación y calculamos las métricas sobre las imágenes reservadas.

No se deben cambiar los parámetros en función de estos resultados; hacerlo convertiría el test en parte del proceso de selección.

In [ ]:
# Reconstruir la configuración ganadora y cargar los mejores pesos guardados.
tf.keras.backend.clear_session()
model = generate_model(num_classes, best_config)
model.set_weights(best_weights)

# Preparar el conjunto de test con la resolución del modelo elegido.
test_ds = make_dataset(
    test_paths, test_labels, best_config["size"], training=False
)

# Evaluar el modelo una sola vez sobre test.
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)

print("Modelo seleccionado:", best_name)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

# Obtener las predicciones para el informe y la matriz de confusión.
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_labels

print("\nInforme de clasificación en test:")
print(classification_report(
    y_true,
    y_pred,
    labels=range(num_classes),
    target_names=class_names,
    zero_division=0
))

# La matriz de confusión muestra aciertos y errores para cada género.
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
).plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False
)
ax.set_title(f"Matriz de confusión en test — {best_name}")
plt.tight_layout()
plt.show()

## 9. Visualizar las curvas de aprendizaje

Finalmente, mostramos las curvas de accuracy y loss de la configuración seleccionada. Sirven para observar cómo evoluciona el entrenamiento y si aparece una diferencia importante entre entrenamiento y validación.

In [ ]:
# Recuperar el historial de la mejor configuración.
selected_history = experiment_results[best_name]["history"]
epochs_range = range(1, len(selected_history["accuracy"]) + 1)

# Gráfica de accuracy.
plt.figure(figsize=(8, 4))
plt.plot(epochs_range, selected_history["accuracy"], label="Train accuracy")
plt.plot(epochs_range, selected_history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title(f"Accuracy — {best_name}")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Gráfica de loss.
plt.figure(figsize=(8, 4))
plt.plot(epochs_range, selected_history["loss"], label="Train loss")
plt.plot(epochs_range, selected_history["val_loss"], label="Validation loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title(f"Loss — {best_name}")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Resumen

- Se han definido 17 configuraciones: `BASE0`, seis experimentos de profundidad y diez experimentos de factores.
- Cada configuración se entrena durante 50 épocas completas.
- La selección del modelo utiliza la accuracy de validación.
- El test se reserva para la evaluación final de una única configuración.
- La configuración de resolución `192×288` se ejecuta al final debido a su mayor coste computacional.

**Antes de ejecutar:** comprueba que `DATA_DIR` apunta a la carpeta que contiene las subcarpetas de géneros musicales.